# 02 tokenizer 和 chat template：文字怎么变成模型能懂的数字

大模型不直接读中文，它读数字。tokenizer 的工作就是把文字切成 token，再映射成 token id。

本项目里 tokenizer 有两个核心用途：推理时构造聊天输入，SFT 训练时构造 prompt 和 full answer。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 推理时的 chat template

In [ ]:
find_lines("scripts/infer.py", "messages = [", context=8)
find_lines("scripts/infer.py", "add_generation_prompt=True", context=8)

`add_generation_prompt=True` 的意思是：在 user 消息后面加上“轮到 assistant 说话了”的模板标记。没有这一步，模型可能不知道接下来该作为 assistant 回答。

## 2. SFT 训练时为什么要分别算 prompt_ids 和 full_ids？

In [ ]:
find_lines("scripts/train_sft_lora.py", "prompt_ids =", context=12)
find_lines("scripts/train_sft_lora.py", "full_ids =", context=12)

这里是理解 SFT 的核心：

- `prompt_ids`：只有 system + user，并且加生成提示。
- `full_ids`：system + user + assistant 标准答案。
- 训练时模型输入 full_ids，但 loss 只算 assistant 部分。

小学生版：题目可以看，但题目不算分；答案才算分。

## 3. 为什么要截断 max_length？

In [ ]:
find_lines("scripts/train_sft_lora.py", "max_length", context=6)

如果一条样本太长，显存会涨，训练会慢，甚至 OOM。这个项目用 512 或更短长度，是为了适配 RTX 4060 Laptop 8GB 显存。

## 4. 看一条真实 SFT 样本

In [ ]:
row = read_jsonl("data/processed/custom_sft_train.jsonl", n=1)[0]
for msg in row["messages"]:
    print("角色:", msg["role"])
    print(msg["content"][:500])
    print("-" * 80)

## 5. 你要能讲出的底层句子

> tokenizer 把中文聊天样本变成 token id；chat template 保证 system/user/assistant 的格式符合 Qwen 的训练习惯；SFT 里会同时构造 prompt token 和 full token，这样才能只对 assistant 答案计算 loss。